In [ ]:
# IMPORT
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def perceptron_simple(x, w, active):
    # Construction de l'entrée augmentée x_tilde = (1, x1, x2)
    x_tilde = np.array([1.0, x[0], x[1]])

    # Calcul du potentiel v = w^T * x_tilde
    v = np.dot(w, x_tilde)

    # Application de la fonction d'activation
    if active == 0:
        # np.sign renvoie -1 si v < 0, +1 si v > 0 (et 1 si v == 0)
        y = 1.0 if v >= 0 else -1.0
    elif active == 1:
        y = np.tanh(v)
    else:
        raise ValueError("Le paramètre 'active' doit valoir 0 (sign) ou 1 (tanh).")

    return float(y)

In [ ]:
# 1. Définition de l'ensemble d'apprentissage du OU logique
X_ou = np.array([
    [0, 0, 1, 1],  # x1
    [0, 1, 0, 1]   # x2
], dtype=float)

# Sorties désirées
yd_ou = np.array([-1, 1, 1, 1])

# Poids donnés : w0 = -0.5, w1 = 1, w2 = 1
w_ou = np.array([-0.5, 1.0, 1.0])

# 2. Test du perceptron sur les 4 exemples avec active = 0
print("--- Test du Perceptron Simple sur le OU logique ---")
for i in range(X_ou.shape[1]):
    x_i = X_ou[:, i]
    y_pred = perceptron_simple(x_i, w_ou, active=0)
    print(f"Entrée x = {x_i} -> Sortie prédite y = {y_pred:+.0f} (Attendu : {yd_ou[i]:+d})")

# 3. Affichage des points et de la droite séparatrice
t = np.array([-0.5, 1.5])                 # abscisses x1 du tracé
z = -(w_ou[0] + w_ou[1] * t) / w_ou[2]    # ordonnées x2 correspondantes

plt.figure(figsize=(6, 6))
plt.plot(t, z, 'k-', label='Droite séparatrice ($w^T\\tilde{x} = 0$)')
plt.plot(X_ou[0, yd_ou == -1], X_ou[1, yd_ou == -1], 'ro', markersize=8, label='Classe -1')
plt.plot(X_ou[0, yd_ou == 1], X_ou[1, yd_ou == 1], 'bx', markersize=8, markeredgewidth=2, label='Classe +1')

plt.xlim(t[0], t[1])
plt.ylim(-0.5, 1.5)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Perceptron simple : OU logique ($w_0=-0.5, w_1=1, w_2=1$)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

In [ ]:
def apprentissage_widrow(x, yd, epoch, batch_size):
    alpha = 0.1
    N = x.shape[1]

    # Ajout de l'entrée constante pour le biais
    X = np.vstack((np.ones(N), x))

    # Initialisation aléatoire des poids
    w = np.random.uniform(-0.5, 0.5, 3)

    erreurs = []

    for e in range(epoch):
        erreur_epoch = 0

        # Mélange des individus à chaque époque
        indices = np.random.permutation(N)

        for debut in range(0, N, batch_size):
            batch = indices[debut:debut + batch_size]

            delta_w = np.zeros(3)

            for i in batch:
                xi = X[:, i]

                # Calcul de la sortie du neurone
                v = np.dot(w, xi)
                y = np.tanh(v)

                # Erreur quadratique
                erreur_epoch += (yd[i] - y) ** 2

                # Règle de Widrow-Hoff
                delta_w += alpha * (yd[i] - y) * (1 - y**2) * xi

            # Mise à jour après chaque lot
            w += delta_w

        erreurs.append(erreur_epoch)

        # Prédictions avec sign
        predictions = np.where(w @ X >= 0, 1, -1)
        nb_erreurs = np.sum(predictions != yd)

        print(f"Époque {e+1} : {nb_erreurs} mal classés")

        # Affichage de la frontière à certaines époques
        if e == 0 or (e + 1) % 10 == 0 or nb_erreurs == 0:
            afficher_frontiere(x, yd, w, e + 1)

        # Arrêt lorsque tous les individus sont bien classés
        if nb_erreurs == 0:
            break

    return w, np.array(erreurs)

In [ ]:
def afficher_frontiere(x, yd, w, epoque):
    plt.figure(figsize=(7, 5))

    plt.scatter(
        x[0, yd == -1],
        x[1, yd == -1],
        label="Classe -1",
        marker="o"
    )

    plt.scatter(
        x[0, yd == 1],
        x[1, yd == 1],
        label="Classe +1",
        marker="x"
    )

    # Droite : w0 + w1*x1 + w2*x2 = 0
    t = np.linspace(x[0].min(), x[0].max(), 100)

    if abs(w[2]) > 1e-12:
        z = -(w[0] + w[1] * t) / w[2]
        plt.plot(t, z, "k-", label="Frontière")
    elif abs(w[1]) > 1e-12:
        plt.axvline(-w[0] / w[1], color="black", label="Frontière")

    plt.xlim(x[0].min() - 0.5, x[0].max() + 0.5)
    plt.ylim(x[1].min() - 0.5, x[1].max() + 0.5)

    plt.title(f"Frontière de décision - Époque {epoque}")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Chargement des données
X = np.loadtxt("p2_d1.txt")

# 25 individus de chaque classe
yd = np.concatenate((-np.ones(25), np.ones(25)))

# Apprentissage online
w_online, erreur_online = apprentissage_widrow(
    X, yd, epoch=100, batch_size=1
)

print("Poids finaux :", w_online)

# Évolution de l'erreur
plt.figure(figsize=(8, 5))
plt.plot(erreur_online)
plt.xlabel("Époque")
plt.ylabel("Erreur quadratique cumulée")
plt.title("Évolution de l'erreur - Test 1 (Online)")
plt.grid(True)
plt.show()

In [ ]:
# Chargement des données
X = np.loadtxt("p2_d1.txt")

# 25 individus de chaque classe
yd = np.concatenate((-np.ones(25), np.ones(25)))

# Apprentissage online
w_online, erreur_online = apprentissage_widrow(
    X, yd, epoch=100, batch_size=1
)

print("Poids finaux :", w_online)

# Évolution de l'erreur
plt.figure(figsize=(8, 5))
plt.plot(erreur_online)
plt.xlabel("Époque")
plt.ylabel("Erreur quadratique cumulée")
plt.title("Évolution de l'erreur - Test 1 (Online)")
plt.grid(True)
plt.show()

In [ ]:
w_batch, erreur_batch = apprentissage_widrow(
    X, yd, epoch=100, batch_size=50
)

plt.figure(figsize=(8, 5))

plt.plot(erreur_online, label="Online (batch_size=1)")
plt.plot(erreur_batch, label="Batch (batch_size=50)")

plt.xlabel("Époque")
plt.ylabel("Erreur quadratique cumulée")
plt.title("Comparaison Online / Batch - Test 1")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
X2 = np.loadtxt("p2_d2.txt")

yd2 = np.concatenate((-np.ones(25), np.ones(25)))

w2, erreur2 = apprentissage_widrow(
    X2, yd2, epoch=100, batch_size=1
)

plt.figure(figsize=(8, 5))

plt.plot(erreur2, label="Test 2")
plt.xlabel("Époque")
plt.ylabel("Erreur quadratique cumulée")
plt.title("Évolution de l'erreur - Test 2")
plt.legend()
plt.grid(True)
plt.show()

In [24]:
# PARTIE 1.3
def sigmoide(v):
    return 1 / (1 + np.exp(-v))

def multiperceptron(x, w1, w2):
    # Construction de l'entrée augmentée x_tilde = (1, x1, x2)
    x_tilde = np.array([1.0, x[0], x[1]])

    # Calcul du potentiel pour la couche caché
    v1 = np.dot(w1[:,0], x_tilde)
    v2 = np.dot(w1[:,1], x_tilde)

    # Application de la fonction d'activation couche caché
    y1 = sigmoide(v1) #resultat neurone 1
    y2 = sigmoide(v2) #resultat neurone 2

    # Entrée augmentée pour la couche de sortie
    y_tilde = np.array([1.0, y1, y2])

    # Potentiel couche de sortie
    v_final = np.dot(w2, y_tilde)

    # Fonction activation couche de sortie
    y_final = sigmoide(v_final)

    return float(y_final)

In [25]:
# Test 1.3

w1 = np.array([
    [-0.5,  0.5],
    [ 2.0,  1.0],
    [-1.0,  0.5]
])

w2 = np.array([2.0, -1.0, 1.0])

x_test = np.array([1.0, 1.0])

y_sortie = multiperceptron(x_test, w1, w2)

print(f"Sortie du Perceptron Multicouche pour l'entrée x = {x_test} : {y_sortie:.5f}")
print("Valeur attendue (calculée à la main) : ~0.90537")

Sortie du Perceptron Multicouche pour l'entrée x = [1. 1.] : 0.90537
Valeur attendue (calculée à la main) : ~0.90537
